# Task3：FLAIR领域编码器与ImageNet ResNet50配对对照

遵循“预训练不要与中央凹定位直接相关”：只用FLAIR基础图文/疾病类别预训练的视觉编码器，不用ZIAN、Toolbox或任何作者已训练的中央凹头。两个定位头都随机初始化，只用本题训练坐标训练。没有嵌入旧预测或测试标签。

来源：[FLAIR论文](https://arxiv.org/abs/2308.07898)、[作者权重](https://huggingface.co/jusiro2/FLAIR/blob/main/model.safetensors)。未完成预训练图片与本题的逐图交集检查，不声称零重叠。当前公开最佳12.31624保持原样。

对照是相同架构、相同随机定位头、相同数据增强/批次/训练规则的ResNet50。两份权重使用各自来源的输入标准化：ImageNet mean/std；FLAIR为[0,1]。因此比较的是两套初始化与预处理方案，不能把差值全部归因于预训练数据。


## 配置与依赖

添加2026task3比赛数据，开启GPU和Internet。本地也可通过同源CLI训练；此Notebook仅用于Kaggle。只使用cuda:0，batch4、CPU2线程，缓存缩小后的图像。

下载约102MB ImageNet和533MB FLAIR包，校验来源哈希；FLAIR只载入视觉部分（约23.5M骨干参数），不创建文本模型。若safetensors未安装，仅安装这个小依赖，不替换Torch版本。


In [ ]:
from pathlib import Path
if not Path('/kaggle/working').is_dir():raise RuntimeError('请用本地CLI启动本机训练；此Notebook入口用于Kaggle。')
DATA_DIR=None
SMOKE_TEST=False
OUTPUT_DIR=Path('/kaggle/working/task3_flair_transfer_v1'+('_smoke' if SMOKE_TEST else ''))
RESUME_DIR=None
WEIGHT_PATHS=None
RUN_FOLDS=[0] if SMOKE_TEST else [0,1,2,3,4]
BATCH_SIZE=4
CREATE_FULL_BACKUP=True
try:
    import safetensors
except ImportError:
    import subprocess,sys
    subprocess.check_call([sys.executable,'-m','pip','install','safetensors>=0.4,<1'])


In [ ]:
from __future__ import annotations
from collections import defaultdict
from contextlib import contextmanager
import argparse, csv, gc, hashlib, json, math, os, random, shutil, statistics, sys, time, zipfile
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

def save_json(path, value):
    path = Path(path)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(4)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def find_data(explicit=None):
    if explicit:
        candidates = [Path(explicit)]
    else:
        base = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
        candidates = [base / '2026task3/detection', Path.cwd() / '2026task3/detection',
                      Path.cwd() / 'task3/2026task3/detection']
        if Path('/kaggle/input').exists():
            candidates += [p.parent for p in Path('/kaggle/input').rglob('fovea_localization_train_GT.csv')]
    valid = sorted({p.resolve() for p in candidates
                    if (p / 'fovea_localization_train_GT.csv').is_file()
                    and (p / 'sample_submission.csv').is_file()
                    and (p / 'train').is_dir() and (p / 'test').is_dir()})
    if len(valid) != 1:
        raise ValueError(f'Expected one dataset; set data_dir explicitly. Found: {valid}')
    return valid[0]


def load_manifest(data):
    labels = pd.read_csv(data / 'fovea_localization_train_GT.csv')
    if list(labels.columns) != ['data', 'Fovea_X', 'Fovea_Y'] or labels.isna().any().any():
        raise ValueError('Invalid training CSV columns or missing values.')
    if labels['data'].duplicated().any():
        raise ValueError('Duplicate training IDs.')
    lookup = labels.set_index('data')
    records = []
    for subset in ['train', 'test']:
        files = sorted(p for p in (data / subset).iterdir()
                       if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        for p in files:
            image_id = int(p.stem)
            with Image.open(p) as im:
                im = im.convert('RGB')
                im.load()
                w, h = im.size
                # Hash decoded pixels, not JPEG metadata. Size participates in the hash.
                digest = hashlib.sha256(f'{w},{h}:'.encode() + im.tobytes()).hexdigest()
            record = dict(image_id=image_id, subset=subset, filename=p.name,
                          width=w, height=h, pixel_sha256=digest)
            if subset == 'train':
                if image_id not in lookup.index:
                    raise ValueError(f'Missing label: {p.name}')
                x, y = lookup.loc[image_id, ['Fovea_X', 'Fovea_Y']].astype(float)
                if not np.isfinite([x, y]).all() or not (0 <= x < w and 0 <= y < h):
                    raise ValueError(f'Coordinate outside image: {p.name}')
                record.update(x=float(x), y=float(y), nx=float(x / w), ny=float(y / h))
                xml_path = data / 'train_location' / f'{image_id:04}.xml'
                if not xml_path.is_file():
                    raise ValueError(f'Missing XML: {xml_path}')
                xml = ET.parse(xml_path).getroot()
                if (int(xml.findtext('size/width')), int(xml.findtext('size/height'))) != (w, h):
                    raise ValueError(f'XML size mismatch: {p.name}')
                objects = [v for v in xml.findall('object') if v.findtext('name').lower() == 'fovea']
                if len(objects) != 1:
                    raise ValueError(f'Expected one Fovea box: {xml_path}')
                box = objects[0].find('bndbox')
                xmin, ymin, xmax, ymax = [float(box.findtext(k)) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
                if not (xmin <= x <= xmax and ymin <= y <= ymax):
                    raise ValueError(f'CSV coordinate outside XML box: {p.name}')
                record.update(xml_center_dx=float(x - (xmin + xmax) / 2),
                              xml_center_dy=float(y - (ymin + ymax) / 2))
            records.append(record)
    manifest = pd.DataFrame(records)
    if set(manifest.loc[manifest.subset == 'train', 'image_id']) != set(labels['data']):
        raise ValueError('Training images and CSV IDs differ.')
    if manifest.image_id.duplicated().any():
        raise ValueError('Image IDs overlap between train/test.')
    cross = manifest.groupby('pixel_sha256').subset.nunique()
    if (cross > 1).any():
        raise ValueError('Identical pixels occur in both train and test; inspect before continuing.')
    return manifest


def assign_folds(train, seed=42, folds=5):
    """Balance exact-duplicate groups across image resolutions; no test labels used."""
    if folds < 2:
        raise ValueError('Need at least two folds.')
    train = train.copy()
    groups = list(train.groupby('pixel_sha256', sort=True))
    rng = np.random.default_rng(seed)
    buckets = defaultdict(list)
    for digest, group in groups:
        row = group.iloc[0]
        buckets[(int(row.width), int(row.height))].append((digest, len(group)))
    total = np.zeros(folds, dtype=int)
    mapping = {}
    for key in sorted(buckets):
        entries = buckets[key]
        rng.shuffle(entries)
        entries.sort(key=lambda pair: -pair[1])
        sizes = np.zeros(folds, dtype=int)
        for digest, count in entries:
            tie = rng.random(folds)
            fold = min(range(folds), key=lambda f: (sizes[f], total[f], tie[f]))
            mapping[digest] = fold
            sizes[fold] += count
            total[fold] += count
    train['fold'] = train.pixel_sha256.map(mapping)
    if train.groupby('pixel_sha256').fold.nunique().max() != 1:
        raise AssertionError('Duplicate group split across folds.')
    return train


class DecoderBlock(nn.Module):
    def __init__(self, incoming, skip, outgoing):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv2d(incoming + skip, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True),
                                    nn.Conv2d(outgoing, outgoing, 3, padding=1, bias=False),
                                    nn.GroupNorm(8, outgoing), nn.ReLU(inplace=True))

    def forward(self, value, skip):
        value = F.interpolate(value, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        return self.layers(torch.cat([value, skip], dim=1))


def heatmap_loss(logits, points, sigma=1.5):
    height, width = logits.shape[-2:]
    yy = torch.arange(height, device=logits.device, dtype=torch.float32) + .5
    xx = torch.arange(width, device=logits.device, dtype=torch.float32) + .5
    distance = (xx[None, None, :] - points[:, 0, None, None] * width) ** 2
    distance = distance + (yy[None, :, None] - points[:, 1, None, None] * height) ** 2
    target = torch.exp(-distance / (2 * sigma ** 2))
    target = target / target.sum(dim=(-2, -1), keepdim=True).clamp_min(1e-12)
    return -(target.flatten(1) * F.log_softmax(logits.float().flatten(1), dim=1)).sum(1).mean()


def decode_heatmap(logits, radius=3):
    """Local expectation around the peak; cell centers match resize align_corners=False."""
    batch, height, width = logits.shape
    peak = logits.flatten(1).argmax(1)
    py, px = peak // width, peak % width
    yy = torch.arange(height, device=logits.device)[None, :, None]
    xx = torch.arange(width, device=logits.device)[None, None, :]
    mask = (yy - py[:, None, None]).abs() <= radius
    mask = mask & ((xx - px[:, None, None]).abs() <= radius)
    weights = logits.float().masked_fill(~mask, -torch.inf).flatten(1).softmax(1).view(batch, height, width)
    x = (weights * ((xx + .5) / width)).sum((-2, -1))
    y = (weights * ((yy + .5) / height)).sum((-2, -1))
    return torch.stack([x, y], dim=1)


@contextmanager
def training_lock(out):
    handle = (out / 'training.lock').open('a+b')
    if handle.tell() == 0:
        handle.write(b'0')
        handle.flush()
    handle.seek(0)
    try:
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        yield
    finally:
        handle.close()


def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':'), allow_nan=False).encode('utf-8')).hexdigest()


def atomic_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    fields = list(fields or (list(rows[0]) if rows else []))
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def read_rows(path):
    with Path(path).open(encoding='utf-8', newline='') as file:
        return list(csv.DictReader(file))


def prediction_rows(records, points, fold, labeled):
    if len(records) != len(points):
        raise ValueError('Prediction count differs from image count.')
    rows, seen = [], set()
    for item, point in zip(records, points):
        key, width, height = int(item['image_id']), int(item['width']), int(item['height'])
        if key in seen:
            raise ValueError('Duplicate image ID.')
        seen.add(key)
        nx, ny = float(point[0]), float(point[1])
        if not all(math.isfinite(v) for v in [nx, ny]) or not (0 <= nx < 1 and 0 <= ny < 1):
            raise ValueError(f'Invalid normalized prediction for image {key}.')
        row = dict(image_id=key, fold=int(fold), width=width, height=height,
                   pixel_sha256=str(item['pixel_sha256']), nx=nx, ny=ny,
                   Fovea_X=nx * width, Fovea_Y=ny * height)
        if labeled:
            x, y = float(item['x']), float(item['y'])
            dx, dy = row['Fovea_X'] - x, row['Fovea_Y'] - y
            row.update(gt_x=x, gt_y=y, dx=dx, dy=dy,
                       distance_px=math.hypot(dx, dy), squared_error=dx * dx + dy * dy)
        rows.append(row)
    return rows


def quantile_linear(values, probability):
    values = sorted(float(v) for v in values)
    if not values:
        raise ValueError('No values.')
    position = (len(values) - 1) * probability
    low, high = math.floor(position), math.ceil(position)
    return values[low] + (values[high] - values[low]) * (position - low)


def metrics_from_rows(rows):
    rows = list(rows)
    if not rows:
        raise ValueError('No labeled predictions.')
    distance, dx, dy = [], [], []
    for row in rows:
        ex = float(row['Fovea_X']) - float(row['gt_x'])
        ey = float(row['Fovea_Y']) - float(row['gt_y'])
        if not math.isfinite(ex) or not math.isfinite(ey):
            raise ValueError('Non-finite error.')
        dx.append(ex)
        dy.append(ey)
        distance.append(math.hypot(ex, ey))
    return dict(n=len(rows), mean_distance_px=statistics.mean(distance),
                median_distance_px=statistics.median(distance),
                p90_distance_px=quantile_linear(distance, .9), max_distance_px=max(distance),
                mae_x_px=statistics.mean(abs(v) for v in dx), mae_y_px=statistics.mean(abs(v) for v in dy),
                rmse_coordinate_px=math.sqrt(sum(x*x+y*y for x, y in zip(dx, dy)) / (2 * len(rows))))


def validate_oof(rows, records):
    expected = {int(r['image_id']): r for r in records}
    if len(expected) != len(records) or len(rows) != len(records):
        raise ValueError('OOF must contain exactly one row per training image.')
    seen, groups = set(), {}
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen:
            raise ValueError('Unknown or repeated OOF image ID.')
        seen.add(key)
        source = expected[key]
        if int(row['fold']) != int(source['fold']):
            raise ValueError('OOF fold does not match the fixed split.')
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('OOF dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('OOF image hash differs.')
        if abs(float(row['gt_x']) - float(source['x'])) > 1e-6 or abs(float(row['gt_y']) - float(source['y'])) > 1e-6:
            raise ValueError('OOF ground truth differs.')
        groups.setdefault(str(source['pixel_sha256']), set()).add(int(row['fold']))
    if seen != set(expected) or any(len(value) != 1 for value in groups.values()):
        raise ValueError('Incomplete OOF coverage or duplicate group leakage.')
    return sorted(rows, key=lambda r: int(r['image_id']))


def validate_test_rows(rows, records, fold):
    expected = {int(r['image_id']): r for r in records}
    if len(rows) != len(expected):
        raise ValueError('Incomplete test prediction coverage.')
    seen = set()
    for row in rows:
        key = int(row['image_id'])
        if key not in expected or key in seen or int(row['fold']) != fold:
            raise ValueError('Invalid test ID/fold.')
        seen.add(key)
        source = expected[key]
        if (int(row['width']), int(row['height'])) != (int(source['width']), int(source['height'])):
            raise ValueError('Test image dimensions differ.')
        if str(row['pixel_sha256']) != str(source['pixel_sha256']):
            raise ValueError('Test image hash differs.')
        x, y = float(row['Fovea_X']), float(row['Fovea_Y'])
        if not all(math.isfinite(v) for v in [x, y]) or not (0 <= x < int(source['width']) and 0 <= y < int(source['height'])):
            raise ValueError('Test prediction outside image.')
    return {int(r['image_id']): r for r in rows}


def average_test_folds(folds, records, fold_count=5):
    if set(folds) != set(range(fold_count)):
        raise ValueError('All five folds are required before exporting a five-fold submission.')
    mappings = [validate_test_rows(folds[f], records, f) for f in range(fold_count)]
    averaged = []
    for source in records:
        key = int(source['image_id'])
        averaged.append(dict(image_id=key, width=int(source['width']), height=int(source['height']),
                             Fovea_X=statistics.mean(float(m[key]['Fovea_X']) for m in mappings),
                             Fovea_Y=statistics.mean(float(m[key]['Fovea_Y']) for m in mappings)))
    return averaged


def write_submission(template_path, predictions, target):
    lookup = {}
    for row in predictions:
        key = int(row['image_id'])
        for axis, dimension in [('X', 'width'), ('Y', 'height')]:
            value = float(row[f'Fovea_{axis}'])
            if not math.isfinite(value) or not 0 <= value < int(row[dimension]):
                raise ValueError('Invalid original-pixel coordinate.')
            image_id = f'{key}_Fovea_{axis}'
            if image_id in lookup:
                raise ValueError('Repeated submission ID.')
            lookup[image_id] = value
    template = read_rows(template_path)
    ids = [row['ImageID'] for row in template]
    if len(ids) != len(set(ids)) or set(ids) != set(lookup):
        raise ValueError('Submission IDs differ from template.')
    rows = [dict(ImageID=key, value=f'{lookup[key]:.6f}') for key in ids]
    atomic_csv(target, rows, ['ImageID', 'value'])
    check = read_rows(target)
    if [r['ImageID'] for r in check] != ids or any(not math.isfinite(float(r['value'])) for r in check):
        raise AssertionError('Submission round trip failed.')


def atomic_torch_save(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(value, temporary)
    temporary.replace(path)


def restore_backup(source, output, protocol):
    if source is None:
        return
    source = Path(source).resolve()
    if source == output.resolve():
        return
    candidate = source / 'protocol.json'
    if not candidate.is_file() or json.loads(candidate.read_text()) != protocol:
        raise ValueError('Backup protocol/data/code differs. Choose the correct prior output folder.')
    if (output / 'protocol.json').exists():
        print('Current output already has progress; resuming it without merging backup files.', flush=True)
        return
    for path in source.rglob('*'):
        if not path.is_file() or path.is_symlink() or path.suffix in {'.tmp', '.zip'}:
            continue
        if not path.resolve().is_relative_to(source):
            raise ValueError('Backup path escapes its source directory.')
        destination = output / path.relative_to(source)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, destination)


In [ ]:
from __future__ import annotations
TRANSFER_SOURCE_SHA256={'task3_pipeline.py': '5a27db4b7901d185eec831080a16d783ad001adc22b639474d345fe7b315ec3f', 'task3_kaggle_cv.py': '2227703a35b3955b8ce02801930b0bf496703decece4ad5234a1d1f70a0b12e6', 'task3_flair_transfer.py': 'ba5659d36899e6dff156deb254703f9eff8c56498130bd358ecb52519d01e787'}
"""FLAIR visual initialization versus ImageNet: independent full-image localization."""
import argparse
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import random
import shutil
import statistics
import sys
import time
import urllib.request
import zipfile


import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

TRANSFER_VERSION='flair-transfer-v1'
RECIPES=('imagenet','flair')
SOURCE_NAMES=('task3_pipeline.py','task3_kaggle_cv.py','task3_flair_transfer.py')
WEIGHT_SPECS={
    'imagenet':dict(filename='resnet50-0676ba61.pth',url='https://download.pytorch.org/models/resnet50-0676ba61.pth',
        sha256_prefix='0676ba61',normalization='imagenet',pretraining='ImageNet-1K IMAGENET1K_V1; no fovea localization checkpoint'),
    'flair':dict(filename='flair_model.safetensors',url='https://huggingface.co/jusiro2/FLAIR/resolve/main/model.safetensors',
        sha256_prefix='050334cd934fa3126435c202c41f493ca222fc3fe3ea9c50cf21f67c792a1440',normalization='zero_one',
        pretraining='Author FLAIR base image-text/categorical pretraining; visual encoder only; no fovea-specific decoder')}


def file_sha(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(8*1024**2),b''):digest.update(chunk)
    return digest.hexdigest()


def get_weights(output,explicit=None):
    folder=Path(output)/'_pretrained';folder.mkdir(parents=True,exist_ok=True);paths={};sources={}
    for recipe,spec in WEIGHT_SPECS.items():
        path=Path(explicit[recipe]) if explicit and explicit.get(recipe) else folder/spec['filename']
        if not path.is_file():
            if explicit and explicit.get(recipe):raise FileNotFoundError(path)
            temporary=path.with_suffix(path.suffix+'.part')
            print(f'Downloading {recipe}: {spec["url"]}',flush=True)
            request=urllib.request.Request(spec['url'],headers={'User-Agent':'Task3 research weight downloader'})
            with urllib.request.urlopen(request,timeout=60) as response,temporary.open('wb') as stream:
                if 'html' in response.headers.get('Content-Type','').lower():raise ValueError('Weight endpoint returned HTML.')
                count=0
                for chunk in iter(lambda:response.read(8*1024**2),b''):
                    stream.write(chunk);count+=len(chunk)
                print(f'{recipe}: downloaded {count/1024**2:.1f} MiB',flush=True)
            if not file_sha(temporary).startswith(spec['sha256_prefix']):raise ValueError('Downloaded checkpoint checksum differs.')
            temporary.replace(path)
        digest=file_sha(path)
        if not digest.startswith(spec['sha256_prefix']):raise ValueError(f'Checkpoint checksum differs: {path}')
        paths[recipe]=path;sources[recipe]=dict(spec,path=str(path.resolve()),sha256=digest,size_bytes=path.stat().st_size)
    save_json(Path(output)/'pretraining_sources.json',sources)
    return paths


def require_safe_open():
    try:from safetensors import safe_open
    except ImportError:
        base=Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
        for folder in (base.parent/'task2/.model_deps',base/'.model_deps'):
            if (folder/'safetensors').is_dir():sys.path.insert(0,str(folder));break
        from safetensors import safe_open
    return safe_open


def backbone_state(recipe,path):
    if recipe=='imagenet':
        state=torch.load(path,map_location='cpu',weights_only=True)
        if set(k for k in state if k.startswith('fc.'))!={'fc.weight','fc.bias'}:raise ValueError('Unexpected ImageNet classifier keys.')
        return {k:v for k,v in state.items() if not k.startswith('fc.')}
    if recipe!='flair':raise ValueError('Unknown recipe.')
    safe_open=require_safe_open();prefix='vision_model.model.'
    with safe_open(path,framework='pt',device='cpu') as tensors:
        state={k[len(prefix):]:tensors.get_tensor(k) for k in tensors.keys() if k.startswith(prefix)}
    if len(state)!=318 or 'conv1.weight' not in state or any(k.startswith('fc.') for k in state):
        raise ValueError('Unexpected FLAIR visual encoder key layout.')
    return state


class TransferHeatmap(nn.Module):
    def __init__(self,recipe=None,weights=None):
        super().__init__();self.backbone=models.resnet50(weights=None);self.backbone.fc=nn.Identity()
        if weights is not None:self.backbone.load_state_dict(backbone_state(recipe,weights),strict=True)
        self.d3=DecoderBlock(2048,1024,128);self.d2=DecoderBlock(128,512,64);self.d1=DecoderBlock(64,256,32)
        self.head=nn.Conv2d(32,1,1)

    def train(self,mode=True):
        super().train(mode)
        for module in self.backbone.modules():
            if isinstance(module,nn.BatchNorm2d):module.eval()
        return self

    def forward(self,image):
        b=self.backbone;x=b.maxpool(b.relu(b.bn1(b.conv1(image))))
        c1=b.layer1(x);c2=b.layer2(c1);c3=b.layer3(c2);c4=b.layer4(c3)
        return self.head(self.d1(self.d2(self.d3(c4,c3),c2),c1))[:,0]


def set_trainable(model,epoch,warm_epochs=5):
    for name,p in model.backbone.named_parameters():p.requires_grad_(epoch>warm_epochs and name.startswith(('layer3.','layer4.')))


def canvas_size(width,height,size=512):
    if min(width,height,size)<=0:raise ValueError('Nonpositive image dimension.')
    scale=size/max(width,height)
    return max(1,int(width*scale)),max(1,int(height*scale))


def to_canvas(point,width,height,size=512):
    nw,nh=canvas_size(width,height,size)
    return float(point[0])/width*nw/size,float(point[1])/height*nh/size


def from_canvas(point,width,height,size=512):
    """Undo canvas padding and return original-image NORMALIZED coordinates."""
    nw,nh=canvas_size(width,height,size)
    return float(point[0])*size/nw,float(point[1])*size/nh


class CanvasCache:
    def __init__(self,data):self.data=Path(data);self.images={}
    def get(self,row):
        key=int(row['image_id'])
        if key not in self.images:
            with Image.open(self.data/row['subset']/row['filename']) as im:
                if im.size!=(int(row['width']),int(row['height'])):raise ValueError('Image dimensions changed.')
                self.images[key]=im.convert('RGB').resize(canvas_size(*im.size),Image.Resampling.BILINEAR)
        return self.images[key].copy()


class CanvasDataset(Dataset):
    def __init__(self,records,cache,cfg,training):
        self.records,self.cache,self.cfg,self.training=list(records),cache,cfg,training
        ids={int(r['image_id']) for r in self.records}
        if len(ids)!=len(self.records):raise ValueError('Repeated image.')
        if training and (ids!=set(cfg['train_ids']) or ids&set(cfg['forbidden_ids'])):raise ValueError('Training images differ from phase.')
        operations=[transforms.ToTensor()]
        if cfg['recipe']=='imagenet':operations.append(transforms.Normalize([.485,.456,.406],[.229,.224,.225]))
        self.tensor=transforms.Compose(operations)
        self.jitter=transforms.ColorJitter(brightness=.15,contrast=.15,saturation=.08,hue=.01)

    def __len__(self):return len(self.records)

    def __getitem__(self,index):
        row=self.records[index];image=self.cache.get(row);nw,nh=image.size
        point=np.array(to_canvas((row['x'],row['y']),int(row['width']),int(row['height'])) if self.training else (nw/1024,nh/1024),dtype=np.float32)
        if self.training:
            point*=512
            if random.random()<.5:image=image.transpose(Image.Transpose.FLIP_LEFT_RIGHT);point[0]=nw-point[0]
            angle,scale=math.radians(random.uniform(-8,8)),random.uniform(.95,1.05)
            a,b=scale*math.cos(angle),-scale*math.sin(angle)
            tx,ty=random.uniform(-.02,.02)*nw,random.uniform(-.02,.02)*nh
            matrix=np.array([[a,b,nw/2+tx-a*nw/2-b*nh/2],[-b,a,nh/2+ty+b*nw/2-a*nh/2],[0.,0.,1.]])
            point=((matrix@[point[0],point[1],1.])[:2]/512).astype(np.float32)
            if not (0<point[0]<nw/512 and 0<point[1]<nh/512):raise ValueError('Augmentation moved target outside content.')
            image=image.transform((nw,nh),Image.Transform.AFFINE,np.linalg.inv(matrix)[:2].reshape(-1),resample=Image.Resampling.BILINEAR,fillcolor=(0,0,0))
            image=self.jitter(image)
        canvas=Image.new('RGB',(512,512));canvas.paste(image,(0,0))
        return self.tensor(canvas),torch.from_numpy(point),torch.tensor([nw,nh],dtype=torch.float32)


def mask_canvas(logits,content):
    h,w=logits.shape[-2:];bounds=content.to(logits.device)
    xx=(torch.arange(w,device=logits.device)+.5)*(512/w)
    yy=(torch.arange(h,device=logits.device)+.5)*(512/h)
    valid=(xx[None,None,:]<bounds[:,0,None,None])&(yy[None,:,None]<bounds[:,1,None,None])
    return logits.float().masked_fill(~valid,-1e4)


def domain_partition(train,fold):
    dev=train[train.fold!=fold].copy();outer=train[train.fold==fold].copy()
    assigned=assign_folds(dev,7000+fold,5);held_ids=set(assigned.loc[assigned.fold==0,'image_id'])
    fit=dev[~dev.image_id.isin(held_ids)].copy();held=dev[dev.image_id.isin(held_ids)].copy()
    groups=[set(f.pixel_sha256) for f in (fit,held,outer)]
    if any(groups[i]&groups[j] for i,j in ((0,1),(0,2),(1,2))) or len(fit)+len(held)!=len(dev):raise ValueError('Group split differs.')
    return dev,fit,held,outer


def phase_config(protocol,fold,recipe,role,training,held,outer_ids,epochs):
    cfg=dict(version=TRANSFER_VERSION,source_sha256=protocol['source_sha256'],data_fingerprint=protocol['data_fingerprint'],recipe=recipe,role=role,outer_fold=fold,
        seed=(52000 if role=='inner' else 53000)+fold,train_ids=list(map(int,training.image_id)),evaluation_ids=list(map(int,held.image_id)) if held is not None else [],
        forbidden_ids=list(outer_ids),epochs=int(epochs),schedule_epochs=protocol['inner_epochs'],warm_epochs=5,image_size=512,batch_size=protocol['batch_size'])
    validate_phase(cfg);return cfg


def validate_phase(cfg):
    sets=[set(cfg[k]) for k in ('train_ids','evaluation_ids','forbidden_ids')]
    if not sets[0] or any(len(s)!=len(cfg[k]) for s,k in zip(sets,('train_ids','evaluation_ids','forbidden_ids'))) or any(sets[i]&sets[j] for i,j in ((0,1),(0,2),(1,2))):raise ValueError('Phase IDs overlap.')
    if cfg['recipe'] not in RECIPES or cfg['role'] not in ('inner','outer') or bool(sets[1])!=(cfg['role']=='inner'):raise ValueError('Phase role differs.')
    if not 1<=cfg['epochs']<=cfg['schedule_epochs'] or cfg['image_size']!=512 or cfg['warm_epochs']!=5:raise ValueError('Phase settings differ.')


@torch.inference_mode()
def predict_transfer(model,loader,fold):
    model.eval();points=[]
    for images,_,content in loader:
        with torch.autocast(device_type='cuda'):logits=model(images.to('cuda:0'))
        if logits.shape!=(len(images),128,128) or not torch.isfinite(logits).all():raise RuntimeError('Invalid heatmap.')
        decoded=decode_heatmap(mask_canvas(logits,content),radius=3).cpu().tolist()
        records=loader.dataset.records[len(points):len(points)+len(images)]
        points.extend(from_canvas(p,int(r['width']),int(r['height'])) for p,r in zip(decoded,records))
    if len(points)!=len(loader.dataset):raise ValueError('Prediction count differs.')
    records=loader.dataset.records;labeled=all(r['subset']=='train' for r in records)
    return prediction_rows(records,points,fold,labeled)


def head_sha(model):
    h=hashlib.sha256()
    for name,tensor in model.state_dict().items():
        if not name.startswith('backbone.'):h.update(name.encode());h.update(tensor.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def gpu_preflight(paths,output):
    results={};head_hashes=[]
    for recipe in RECIPES:
        seed_everything(54000);model=TransferHeatmap(recipe,paths[recipe]).to('cuda:0');head_hashes.append(head_sha(model))
        image=torch.randn(1,3,512,512,device='cuda:0');target=torch.tensor([[.5,.3]],device='cuda:0');content=torch.tensor([[512.,342.]],device='cuda:0')
        checks={}
        for epoch in (1,6):
            set_trainable(model,epoch);model.train();model.zero_grad(set_to_none=True)
            logits=model(image);loss=heatmap_loss(mask_canvas(logits,content),target,2.4);loss.backward()
            if logits.shape!=(1,128,128) or not torch.isfinite(loss):raise RuntimeError('GPU preflight shape/loss failure.')
            if not all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters()):raise RuntimeError('GPU preflight non-finite gradient.')
            if epoch==1 and any(p.grad is not None for p in model.backbone.parameters()):raise RuntimeError('Warm phase updated backbone.')
            if epoch==6:
                for name in ('layer3.0.conv1.weight','layer4.0.conv1.weight'):
                    p=dict(model.backbone.named_parameters())[name]
                    if p.grad is None or not p.grad.abs().sum()>0:raise RuntimeError('Partial fine-tune has no gradient.')
            if any(m.training for m in model.backbone.modules() if isinstance(m,nn.BatchNorm2d)):raise RuntimeError('Backbone BN is not frozen.')
            checks[str(epoch)]=float(loss.detach())
        results[recipe]=checks
        del model,image,target,content,logits,loss;gc.collect();torch.cuda.empty_cache()
    if len(set(head_hashes))!=1:raise RuntimeError('The paired random localization heads differ.')
    save_json(Path(output)/'gpu_preflight.json',dict(passed=True,head_sha256=head_hashes[0],losses=results))


def train_phase(records,cache,cfg,folder,weight_path,evaluator=None):
    validate_phase(cfg)
    if (evaluator is not None)!=(cfg['role']=='inner'):raise ValueError('Outer evaluator is forbidden.')
    folder.mkdir(parents=True,exist_ok=True);signature=stable_hash(cfg)
    if (folder/'config.json').exists() and json.loads((folder/'config.json').read_text())!=cfg:raise ValueError('Configuration differs.')
    save_json(folder/'config.json',cfg);checkpoint=folder/('best.pt' if evaluator else 'final.pt')
    if (folder/'phase_complete.json').exists():
        done=json.loads((folder/'phase_complete.json').read_text())
        if done['signature']!=signature or not checkpoint.is_file():raise ValueError('Completed phase requires full weights.')
        return done
    seed_everything(cfg['seed']);last=folder/'last.pt';model=TransferHeatmap(cfg['recipe'],None if last.exists() else weight_path).to('cuda:0')
    save_json(folder/'initial_head.json',dict(seed=cfg['seed'],sha256=head_sha(model)))
    ids={id(p) for p in model.backbone.parameters()}
    optimizer=torch.optim.AdamW([dict(params=model.backbone.parameters(),lr=1e-5),dict(params=[p for p in model.parameters() if id(p) not in ids],lr=5e-4)],weight_decay=1e-4)
    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lambda step:.05+.95*(1+math.cos(math.pi*step/cfg['schedule_epochs']))/2)
    scaler=torch.amp.GradScaler('cuda');start=0;history=[];best=dict(score=float('inf'),epoch=0);best_model=None
    if last.exists():
        state=torch.load(last,map_location='cpu',weights_only=False)
        if state['signature']!=signature:raise ValueError('Resume signature differs.')
        model.load_state_dict(state['model'],strict=True);optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        start,history,best,best_model=state['epoch'],state['history'],state['best'],state['best_model'];del state
        if evaluator and best['epoch']:atomic_torch_save(checkpoint,dict(model=best_model,signature=signature,epoch=best['epoch'],config=cfg,selection=best))
    loader=DataLoader(CanvasDataset(records,cache,cfg,True),batch_size=cfg['batch_size'],shuffle=True,num_workers=0)
    for epoch in range(start+1,cfg['epochs']+1):
        seed_everything(cfg['seed']+1000*epoch);torch.set_num_threads(2);set_trainable(model,epoch);model.train();begin=time.perf_counter();total=count=skips=0
        for images,points,content in loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda'):logits=model(images.to('cuda:0'));loss=heatmap_loss(mask_canvas(logits,content),points.to('cuda:0'),2.4)
            if not torch.isfinite(loss):raise RuntimeError('Non-finite training loss.')
            scaler.scale(loss).backward();scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),5.)
            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update();skips+=int(scaler.get_scale()<old_scale)
            total+=float(loss.detach())*len(images);count+=len(images)
        if skips==len(loader) and history and history[-1]['amp_skips']==len(loader):raise RuntimeError('All optimizer updates skipped in consecutive epochs.')
        row=dict(epoch=epoch,train_ce=total/count,amp_skips=skips,backbone_stage='head_only' if epoch<=5 else 'layer3_layer4',lr_backbone=optimizer.param_groups[0]['lr'])
        if evaluator:
            metrics=evaluator(model);score=metrics['rmse_coordinate_px'];row.update(metrics)
            if not math.isfinite(score):raise RuntimeError('Non-finite validation RMSE.')
            if score<best['score']:
                best=dict(score=score,epoch=epoch);best_model={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                atomic_torch_save(checkpoint,dict(model=best_model,signature=signature,epoch=epoch,config=cfg,selection=best))
        scheduler.step();row['seconds']=time.perf_counter()-begin;history.append(row)
        if epoch%5==0 or epoch==cfg['epochs']:
            atomic_torch_save(last,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict(),signature=signature,epoch=epoch,history=history,best=best,best_model=best_model))
        atomic_csv(folder/'history.csv',history);save_json(folder/'status.json',dict(state='training',epoch=epoch,signature=signature))
        print(f'Fold{cfg["outer_fold"]} {cfg["role"]}/{cfg["recipe"]} {epoch:02}/{cfg["epochs"]} CE={total/count:.4f}'+(f' RMSE={score:.3f} best={best["score"]:.3f}' if evaluator else '')+f' {row["seconds"]:.1f}s',flush=True)
    if not evaluator:atomic_torch_save(checkpoint,dict(model=model.state_dict(),signature=signature,epoch=cfg['epochs'],config=cfg))
    done=dict(signature=signature,epochs=cfg['epochs'],best=best if evaluator else {});save_json(folder/'phase_complete.json',done)
    if last.exists():last.unlink()
    del model,optimizer,scaler,loader,best_model;gc.collect();torch.cuda.empty_cache();return done


def fold_design(prepared,fold):
    dev,fit,held,outer=domain_partition(prepared['train'],fold)
    design=dict(protocol=prepared['protocol'],fold=fold,fit_ids=list(map(int,fit.image_id)),inner_val_ids=list(map(int,held.image_id)),outer_ids=list(map(int,outer.image_id)))
    return dev,fit,held,outer,design


def run_fold(prepared,fold,paths,cache):
    dev,fit,held,outer,design=fold_design(prepared,fold);job=prepared['output']/f'fold_{fold}';job.mkdir(parents=True,exist_ok=True);signature=stable_hash(design)
    if (job/'fold_complete.json').exists():
        done=json.loads((job/'fold_complete.json').read_text())
        if done['signature']!=signature or any(not (job/p).is_file() for p in done['required_models']):raise ValueError('Completed fold needs full matching weights.')
        return
    save_json(job/'design.json',design);decisions={}
    atomic_csv(job/'split_roles.csv',[dict(image_id=int(r['image_id']),pixel_sha256=r['pixel_sha256'],role=role) for role,frame in [('inner_train',fit),('inner_validation',held),('outer_validation',outer)] for r in frame.to_dict('records')])
    for recipe in RECIPES:
        cfg=phase_config(prepared['protocol'],fold,recipe,'inner',fit,held,design['outer_ids'],prepared['protocol']['inner_epochs'])
        validation=DataLoader(CanvasDataset(held.to_dict('records'),cache,cfg,False),batch_size=cfg['batch_size'],shuffle=False,num_workers=0)
        def evaluate(model):return metrics_from_rows(predict_transfer(model,validation,-1))
        done=train_phase(fit.to_dict('records'),cache,cfg,job/f'inner_{recipe}',paths[recipe],evaluate)
        decisions[recipe]=done['best'];del validation
        cfg=phase_config(prepared['protocol'],fold,recipe,'outer',dev,None,design['outer_ids'],done['best']['epoch'])
        folder=job/f'outer_{recipe}';train_phase(dev.to_dict('records'),cache,cfg,folder,paths[recipe])
        state=torch.load(folder/'final.pt',map_location='cpu',weights_only=True)
        if state['signature']!=stable_hash(cfg):raise ValueError('Inference signature differs.')
        model=TransferHeatmap().to('cuda:0');model.load_state_dict(state['model'],strict=True);del state
        for subset,frame in [('validation',outer),('test',prepared['test'])]:
            loader=DataLoader(CanvasDataset(frame.to_dict('records'),cache,cfg,False),batch_size=cfg['batch_size'],shuffle=False,num_workers=0)
            atomic_csv(job/f'{subset}_{recipe}.csv',predict_transfer(model,loader,fold));del loader
        del model;gc.collect();torch.cuda.empty_cache()
    required=sorted(p.relative_to(job).as_posix() for p in job.rglob('*.pt'))
    save_json(job/'fold_complete.json',dict(signature=signature,decisions=decisions,required_models=required,predictions_complete=True,outer_labels_used_for_epoch_selection=False))


def reports(prepared):
    output=prepared['output'];folds=[f for f in range(5) if (output/f'fold_{f}/fold_complete.json').exists()]
    selection=dict(complete=len(folds)==5 and not prepared['protocol']['smoke_test'],completed_folds=folds,smoke_test=prepared['protocol']['smoke_test'],
        metric='rmse_coordinate_px',candidate_rule='Predeclared FLAIR single-backbone five-fold average',selected_by_outer_scores=False,candidate_public_score=None,
        limitation='Pretraining image overlap not fully audited. Initialization recipes also use their source-specific normalization. The same80 images have previously been used for development.')
    if not selection['complete']:save_json(output/'selection.json',selection);return False
    records,testing=prepared['train'].to_dict('records'),prepared['test'].to_dict('records');oof={};tests={};comparisons=[]
    for recipe in RECIPES:
        preds=[];test_folds={}
        for f in range(5):
            job=output/f'fold_{f}';done=json.loads((job/'fold_complete.json').read_text());design=fold_design(prepared,f)[-1]
            if done['signature']!=stable_hash(design) or not done['predictions_complete'] or done['outer_labels_used_for_epoch_selection']:raise ValueError('Fold provenance differs.')
            outer=prepared['train'][prepared['train'].fold==f].to_dict('records');val=validate_oof(read_rows(job/f'validation_{recipe}.csv'),outer)
            validate_test_rows(val,outer,f);preds.extend(val);test_folds[f]=read_rows(job/f'test_{recipe}.csv');validate_test_rows(test_folds[f],testing,f)
        oof[recipe]=validate_oof(preds,records);tests[recipe]=average_test_folds(test_folds,testing)
        atomic_csv(output/f'oof_{recipe}.csv',oof[recipe]);atomic_csv(output/f'test_{recipe}.csv',tests[recipe]);write_submission(prepared['data']/'sample_submission.csv',tests[recipe],output/f'submission_{recipe}_5fold.csv')
        comparisons.append(dict(model=recipe,**metrics_from_rows(oof[recipe])))
    # A fixed two-model equal blend is diagnostic; it never replaces the declared FLAIR candidate automatically.
    for labeled,groups,target in [(True,oof,'oof_equal.csv'),(False,tests,'test_equal.csv')]:
        maps={r:{int(x['image_id']):x for x in group} for r,group in groups.items()};blended=[]
        for key in sorted(maps['flair']):
            row=dict(maps['flair'][key]);x,y=[statistics.mean(float(maps[r][key][f'Fovea_{a}']) for r in RECIPES) for a in ('X','Y')]
            row.update(Fovea_X=x,Fovea_Y=y,nx=x/int(row['width']),ny=y/int(row['height']))
            if labeled:
                dx,dy=x-float(row['gt_x']),y-float(row['gt_y']);row.update(dx=dx,dy=dy,distance_px=math.hypot(dx,dy),squared_error=dx*dx+dy*dy)
            blended.append(row)
        atomic_csv(output/target,blended)
        if labeled:comparisons.append(dict(model='fixed_equal',**metrics_from_rows(blended)))
        else:write_submission(prepared['data']/'sample_submission.csv',blended,output/'submission_equal_5fold.csv')
    atomic_csv(output/'comparison.csv',comparisons);shutil.copy2(output/'submission_flair_5fold.csv',output/'submission_flair_candidate.csv')
    selection.update(candidate='submission_flair_candidate.csv',public_best_reference=12.31624);save_json(output/'selection.json',selection)
    print(pd.DataFrame(comparisons).to_string(index=False),flush=True);return True


def archive_results(output,full=False):
    target=output.parent/'task3_flair_transfer_full_backup.zip' if full else output/'task3_flair_transfer_results_light.zip';temporary=target.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as archive:
        for p in sorted(output.rglob('*')):
            if not p.is_file() or p.is_symlink() or '_pretrained' in p.relative_to(output).parts or p.suffix in ('.zip','.tmp','.part'):continue
            if full or p.suffix in ('.csv','.json','.md'):archive.write(p,p.relative_to(output))
    temporary.replace(target)
    if full:
        with zipfile.ZipFile(target) as archive:
            names=set(archive.namelist())
            for f in range(5):
                done=json.loads((output/f'fold_{f}/fold_complete.json').read_text())
                if any(f'fold_{f}/{name}' not in names for name in done['required_models']):raise RuntimeError('Backup missing model.')
    return target


def prepare(data_dir=None,output_dir=None,resume_dir=None,smoke=False,batch_size=4):
    if not torch.cuda.is_available():raise RuntimeError('CUDA GPU required; no CPU training fallback.')
    data,output=find_data(data_dir),Path(output_dir).resolve();manifest=load_manifest(data)
    train=assign_folds(manifest[manifest.subset=='train'],42,5);test=manifest[manifest.subset=='test'].copy()
    if len(train)!=80 or len(test)!=20:raise ValueError('Expected80 training and20 test images.')
    fingerprint=[]
    for row in manifest.to_dict('records'):
        r={k:row[k] for k in ('image_id','subset','filename','width','height','pixel_sha256')}
        if row['subset']=='train':r.update(x=float(row['x']),y=float(row['y']))
        fingerprint.append(r)
    hashes=dict(TRANSFER_SOURCE_SHA256) if 'TRANSFER_SOURCE_SHA256' in globals() else {name:file_sha(Path(__file__).resolve().parent/name) for name in SOURCE_NAMES}
    protocol=dict(version=TRANSFER_VERSION,source_sha256=hashes,data_fingerprint=stable_hash(fingerprint),weight_specs=WEIGHT_SPECS,
        inner_epochs=2 if smoke else 40,warm_epochs=5,batch_size=int(batch_size),image_size=512,geometry='aspect-preserving top-left canvas; padded heatmap cells masked',
        adaptation='head warmup5; then layer3/layer4 + decoder; backbone BN frozen',candidate='flair',smoke_test=bool(smoke),
        task_specific_pretraining=False,pretraining_image_overlap_audited=False)
    output.mkdir(parents=True,exist_ok=True)
    if shutil.disk_usage(output).free<9*1024**3:raise RuntimeError('Need9 GiB free for checkpoints and backup.')
    restore_backup(resume_dir,output,protocol)
    if (output/'protocol.json').exists() and json.loads((output/'protocol.json').read_text())!=protocol:raise ValueError('Output protocol differs; choose a new directory.')
    save_json(output/'protocol.json',protocol);train.to_csv(output/'split.csv',index=False);manifest.to_csv(output/'manifest.csv',index=False)
    save_json(output/'environment.json',dict(torch=str(torch.__version__),numpy=np.__version__,gpu=torch.cuda.get_device_name(0),device='cuda:0',cpu_threads=2))
    return dict(data=data,output=output,train=train,test=test,protocol=protocol)


def run(prepared,run_folds=(0,1,2,3,4),weight_paths=None,full_backup=True):
    if not run_folds or len(set(run_folds))!=len(run_folds) or not set(run_folds)<=set(range(5)):raise ValueError('Invalid fold schedule.')
    output=prepared['output'];torch.set_num_threads(2);cache=CanvasCache(prepared['data'])
    pending=any(not (output/f'fold_{f}/fold_complete.json').exists() for f in run_folds)
    with training_lock(output):
        try:
            paths=get_weights(output,weight_paths) if pending else {}
            if pending:gpu_preflight(paths,output)
            for f in run_folds:
                save_json(output/'status.json',dict(state='running',fold=f,updated_unix=time.time()))
                run_fold(prepared,f,paths,cache);reports(prepared);archive_results(output)
            complete=reports(prepared);save_json(output/'status.json',dict(state='completed' if complete else 'partial_completed',updated_unix=time.time()));archive_results(output)
            if complete and full_backup:print('Full backup:',archive_results(output,True),flush=True)
        except BaseException as error:
            save_json(output/'status.json',dict(state='interrupted' if isinstance(error,KeyboardInterrupt) else 'failed',error=str(error),updated_unix=time.time()));archive_results(output);raise
    return output


def cli():
    parser=argparse.ArgumentParser();parser.add_argument('--train',action='store_true');parser.add_argument('--download-only',action='store_true')
    parser.add_argument('--output-dir',required=True);parser.add_argument('--data-dir');parser.add_argument('--resume-dir');parser.add_argument('--weights-dir');parser.add_argument('--smoke',action='store_true');parser.add_argument('--folds',nargs='+',type=int,default=[0,1,2,3,4]);parser.add_argument('--batch-size',type=int,default=4);parser.add_argument('--no-full-backup',action='store_true')
    args=parser.parse_args()
    if args.download_only:
        out=Path(args.output_dir);out.mkdir(parents=True,exist_ok=True);get_weights(out);return
    prepared=prepare(args.data_dir,args.output_dir,args.resume_dir,args.smoke,args.batch_size)
    paths={r:Path(args.weights_dir)/s['filename'] for r,s in WEIGHT_SPECS.items()} if args.weights_dir else None
    run(prepared,[0] if args.smoke else args.folds,weight_paths=paths,full_backup=not args.no_full_backup)


## 训练与独立外层评价

整图保留纵横比缩放到最长边512，贴到左上角512画布；预测时屏蔽补边区域，解码后逆变换回原图坐标。两组采用相同几何处理、翻转/仿射/颜色增强，直接输出整图热图，不依赖旧粗预测或精修器。

每个外层折64图开发/16图评价。64图内部按像素重复组分为约51～52训练/12～13验证，训练40轮，按内层官方坐标RMSE选轮次。随后重新初始化并用64图训练所选轮数，最后评估外层16图。0039/0077不会跨边界，外层评价不用于早停。

每阶段前5轮只训练定位头；之后只解冻骨干layer3/layer4，保持骨干BN冻结。外层重训保留40轮余弦周期，不把选中的较短轮数当作新周期。两组每轮数据随机数配对，定位头初始化相同。

这是新方案的内层早停/外层评价；但本题80张图已经被反复用于开发，OOF不保证私有榜提升。首轮默认最多20个训练阶段，本版实际耗时见日志。


In [ ]:
PREPARED=prepare(DATA_DIR,OUTPUT_DIR,RESUME_DIR,SMOKE_TEST,BATCH_SIZE)
print('GPU:',torch.cuda.get_device_name(0))
for f in RUN_FOLDS:
    dev,fit,held,outer=domain_partition(PREPARED['train'],f)
    print(f'Fold{f}: fit={len(fit)}, inner validation={len(held)}, refit={len(dev)}, outer={len(outer)}')
RESULT_DIR=run(PREPARED,RUN_FOLDS,WEIGHT_PATHS,CREATE_FULL_BACKUP)


## 输出与保存

预先指定候选为FLAIR单骨干五折平均，不根据外层排名自动改选。另存ImageNet对照和固定两模型等权诊断。先交回轻量ZIP核对OOF，再考虑比赛提交。Notebook不自动提交比赛。

完整备份包含20份训练模型权重及日志，不含可重新下载的原始预训练包；每折required_models校验完整性。每5轮保存optimizer及对应最佳权重，避免中断时混入未提交的未来最佳权重。恢复需要本版完整Output、同一数据与代码配置。

SMOKE_TEST仅第0折、每阶段2轮，使用独立目录，禁止正式候选。正式五折全部完成后才生成40行候选。


In [ ]:
from IPython.display import display
print((RESULT_DIR/'selection.json').read_text(encoding='utf-8'))
if (RESULT_DIR/'comparison.csv').exists():display(pd.read_csv(RESULT_DIR/'comparison.csv'))
print('轻量结果:',RESULT_DIR/'task3_flair_transfer_results_light.zip')
print('完整备份:',RESULT_DIR.parent/'task3_flair_transfer_full_backup.zip')
candidate=RESULT_DIR/'submission_flair_candidate.csv'
if candidate.exists():display(pd.read_csv(candidate))
